In [ ]:
%load_ext autoreload
%autoreload 2

# 09 — Repository Onboarding

Welcome to the **Smartflat** project. This notebook walks you through the project setup, repository structure, and key concepts so you can start contributing quickly.

> **Audience:** New lab members joining the Smartflat project.  
> **Prerequisites:** Python >= 3.10, access to the `data-gold-final` dataset.

## 1. Scientific Context

**Smartflat** is a multimodal video analysis framework for healthcare, developed as part of the **SDS2 study** (*Smartflat for Dysexecutive Syndrome Stratification*) at Hôpital Percy, France.

### Study design

| | |
|---|---|
| **Cohort** | 162 administrations from 122 unique participants |
| **Groups** | 26 healthy controls, 59 traumatic brain injury (TBI), 37 right-hemisphere stroke (RIL) |
| **Task** | Cooking task (baking a chocolate cake following a recipe) — a naturalistic assessment of activities of daily living |
| **Recording** | 3 GoPro cameras (60 fps) + Tobii Pro Glasses 2 (eye-tracking at 100 Hz, video at 25 fps) |
| **Additional modalities** | Hand landmarks (MediaPipe), body pose (MediaPipe), speech transcription (WhisperX) |

The goal is to detect and characterize **dysexecutive syndromes** through computational behavioral analysis of the cooking task recordings.

## 2. Installation

```bash
# Clone the repository
git clone <repo-url>
cd smartflat

# Install in development mode (all modalities + dev tools)
pip install -e ".[all,dev]"

# Set up pre-commit hooks (strips notebook outputs before commits)
pre-commit install
```

For a lighter install, pick only the extras you need (see `README.md` for the full list: `pose`, `audio`, `video`, etc.).

## 3. Data Root Configuration

Smartflat needs to know where the `data-gold-final` directory lives. Set the environment variable:

```bash
export SMARTFLAT_DATA_ROOT=/path/to/data-gold-final
```

Common locations:

| Machine | Path |
|---------|------|
| Lab MacBook (external drive) | `/Volumes/Smartflat/data-gold-final` |
| HPC (Ruche) | `/gpfs/workdir/perochons/data-gold-final` |
| Lab desktop (PCNomad) | `/media/sam/Smartflat/data-gold-final` |

Add the export to your shell profile (`~/.zshrc` or `~/.bashrc`) so it persists across sessions.

In [ ]:
# Verify your setup
from smartflat.utils.utils_paths import get_data_root

data_root = get_data_root()
print(f"Data root: {data_root}")


## 4. Repository Structure

```
smartflat/
├── smartflat/              # Python package (library code)
│   ├── configs/            # Configuration classes (BaseConfig + JSON serialization)
│   ├── datasets/           # PyTorch Dataset classes + metadata management
│   ├── features/           # Feature extraction pipelines
│   │   ├── video/          #   VideoMAE-v2 (D=1408)
│   │   ├── audio/          #   WhisperX ASR + multilingual-e5-large
│   │   ├── hands/          #   MediaPipe hand landmarks (21 keypoints)
│   │   ├── hands_processing/  # Hand tracking + join/discard filtering
│   │   ├── skeleton/       #   MediaPipe body pose (33 keypoints)
│   │   ├── gaze/           #   Tobii eye-tracking (incomplete)
│   │   ├── symbolization/  #   Recursive prototyping (Ch. 5)
│   │   └── symbolic_barycenter/  # TWE distance + DBA (Ch. 6)
│   ├── engine/             # Analysis engines (clustering, change-point detection)
│   └── utils/              # Utilities (paths, visualization, clinical)
├── notebooks/              # Exploration and analysis notebooks (NB00–NB10)
├── associated-papers/      # Thesis LaTeX sources (chapters 4–6) + published paper
└── tests/                  # Test suite
```

**Key separation:** `smartflat/` contains reusable library code (imported via `from smartflat.xxx import yyy`). `notebooks/` are for exploration and analysis. Never use `sys.path` manipulation.

## 5. Data Organization

Data follows a **task / participant / modality** hierarchy:

```
data-gold-final/
├── cuisine/                                    # Task name
│   ├── G100_P86_BAUVin_25112022/               # G{id}_P{participant}_{trigram}_{date}
│   │   ├── Tobii/                              # Modality folders
│   │   ├── GoPro1/
│   │   ├── GoPro2/
│   │   ├── GoPro3/
│   │   └── Annotation/
│   └── ...
├── dataframes/                                 # Processed metadata
├── experiments/                                # Experimental results
└── outputs/                                    # Feature extraction outputs
```

Naming convention: `G{global_id}_P{participant_id}_{trigram}_{date}`

In [ ]:
import os

# Explore the data hierarchy
data_root = get_data_root()

# List task folders
task_folders = [d for d in os.listdir(data_root) if os.path.isdir(os.path.join(data_root, d)) and d in ('cuisine', 'lego')]
print(f"Tasks: {task_folders}")

# Show a sample participant
if 'cuisine' in task_folders:
    cuisine_path = os.path.join(data_root, 'cuisine')
    participants = sorted([d for d in os.listdir(cuisine_path) if d.startswith('G')])[:5]
    print(f"\nSample participants (cuisine): {participants}")
    
    if participants:
        sample = os.path.join(cuisine_path, participants[0])
        modalities = os.listdir(sample)
        print(f"Modalities in {participants[0]}: {modalities}")


## 6. Key Concepts

### Metadata-driven processing

All operations iterate over **metadata DataFrames** (not the file tree). Each row represents one video with columns like `identifier`, `task_name`, `participant_id`, `modality`, `video_path`, `fps`, etc.

### Flag-based tracking

Computed features are tracked via hidden **flag files** (e.g., `.merged_video_hand_landmarks_flag.txt`). Each flag contains `success`, `failure`, or is absent (`unprocessed`). The dataset classes use these flags to filter what needs processing.

### Config system

All experiment parameters live in **config classes** (e.g., `BaseSmartflatConfig`) with JSON serialization for reproducibility. Never hardcode parameters in processing scripts.

### Host-aware paths

`get_data_root()` returns different data paths based on the current machine hostname. Set `SMARTFLAT_DATA_ROOT` to override.

In [ ]:
from smartflat.datasets.loader import get_dataset

# Load the base dataset and inspect metadata
dset = get_dataset(dataset_name='base', scenario='all')
print(f"Total entries: {len(dset)}")
print(f"\nColumns: {list(dset.metadata.columns)}")
display(dset.metadata.head(3))

# Summary by task and modality
print("\nEntries by task and modality:")
display(dset.metadata.groupby(['task_name', 'modality']).size().unstack(fill_value=0))


## 7. Feature Extraction Pipelines

Smartflat extracts features from multiple modalities:

| Modality | Model | Output | Module |
|----------|-------|--------|--------|
| **Video** | VideoMAE-v2 (ViT-Giant, D=1408) | Latent representations per 16-frame segment | `features/video/` |
| **Speech** | WhisperX + multilingual-e5-large | ASR transcripts + text embeddings | `features/audio/` |
| **Hands** | MediaPipe Hands (21 landmarks) | Per-frame 3D hand coordinates + tracking | `features/hands/`, `features/hands_processing/` |
| **Skeleton** | MediaPipe Pose (33 landmarks) | Per-frame 3D body joint coordinates | `features/skeleton/` |
| **Gaze** | Tobii Pro Glasses 2 (100 Hz) | Eye-tracking data (incomplete) | `features/gaze/` |

Each pipeline follows the same pattern: load dataset → iterate over unprocessed entries → extract features → save output + flag.

## 8. Notebook Roadmap

| # | Notebook | What it does | Thesis Ch. |
|---|----------|-------------|------------|
| 00 | `00_data_overview.ipynb` | Cohort statistics and dataset summary | 4 |
| 01 | `01_data_preprocessing.ipynb` | Raw recordings to consolidated metadata | 4 |
| 02 | `02_feature_extraction.ipynb` | VideoMAE-v2, WhisperX, MediaPipe extraction | 4 |
| 03 | `03_recursive_prototyping.ipynb` | Cosine k-means clustering and prototype annotation | 5 |
| 04 | `04_temporal_segmentation.ipynb` | Kernel change-point detection (PELT) | 5 |
| 05 | `05_symbolic_representation.ipynb` | Prototypes + segments → symbolic sequences | 5 |
| 06 | `06_barycenter_averaging.ipynb` | Temporal-Wasserstein distance and DBA | 6 |
| 07 | `07_clinical_analysis.ipynb` | Group comparisons: Control vs TBI vs RIL | 6 |
| 08 | `08_thesis_figures.ipynb` | Thesis figure reproduction | — |
| **09** | **`09_onboarding_repository.ipynb`** | **This notebook** | — |
| **10** | **`10_onboarding_hand_landmarks.ipynb`** | **Hand landmarks deep-dive** | 4 |

Start with NB00 for an overview, then proceed in order to follow the full pipeline.

## 9. Where to Go Next

- **[CONTRIBUTING.md](../CONTRIBUTING.md)** — Code style, development workflow, how to add new features
- **[CLAUDE.md](../CLAUDE.md)** — Developer reference: architecture, package structure, configuration patterns
- **[DATA_INVENTORY.md](../DATA_INVENTORY.md)** — Full directory structure of `data-gold-final`
- **[QUICKSTART.md](../QUICKSTART.md)** — Condensed setup instructions

If you're working on **hand landmarks analysis**, proceed directly to **NB10** (`10_onboarding_hand_landmarks.ipynb`).